# Session 8 — Streaming Responses

Streaming is one of the key differences between a static notebook demo and a responsive application. This notebook shows the event pattern we will later reuse in Gradio and Streamlit.

## Learning Goals

- understand why streaming improves app experience
- use the Responses API with `stream=True`
- process text deltas from the event stream
- connect the same pattern to later UI examples
- compare streaming patterns across OpenAI and Ollama


In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_ORG_ID = os.getenv('OPENAI_ORG_ID')
OPENAI_PROJECT_ID = os.getenv('OPENAI_PROJECT_ID')
print('OpenAI key present:', bool(OPENAI_API_KEY))
print('OpenAI org ID present:', bool(OPENAI_ORG_ID))
print('OpenAI project ID present:', bool(OPENAI_PROJECT_ID))

GENERATION_MODEL = os.getenv("SESSION8_MODEL", "gpt-4.1-mini")


OpenAI key present: True
OpenAI org ID present: True
OpenAI project ID present: True


## Non-Streaming Baseline

Start with a normal Response API call so we can compare it to the streaming version.

In [3]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)

response = client.responses.create(
    model=GENERATION_MODEL,
    instructions='You are a concise teaching assistant.',
    input='write a 2 pages explaniation of why  streaming is useful in chat apps.'
)

display(Markdown(response.output_text))

**The Usefulness of Streaming in Chat Applications**

In the realm of modern communication, chat applications play a crucial role in connecting people instantly across the globe. One technological advancement that significantly enhances the user experience in chat apps is streaming. Streaming, in this context, refers to the continuous transmission of data—such as text, audio, video, or multimedia—in real-time, allowing information to be sent and received without waiting for the entire data package to be ready. This paper explores why streaming is especially useful in chat applications by examining its impact on responsiveness, user experience, resource efficiency, and new feature enablement.

### 1. Real-Time Interaction and Responsiveness

A core feature of chat applications is the ability to facilitate real-time conversations. Unlike traditional send-and-receive models where messages are transmitted in discrete chunks, streaming allows data to be sent incrementally as it is generated. This results in near-instantaneous message delivery with minimal latency. For example, streaming enables partial message display as users type or generate replies, which is evident in features like typing indicators or live transcription.

The immediate feedback that streaming provides makes conversations smoother and more natural by reducing delays. Users feel as though they are speaking face-to-face because messages appear almost instantly, fostering engagement and effective communication. This is particularly important in group chats where timely participation ensures fluid discussion flow.

### 2. Enhanced User Experience

Streaming data in chat apps also improves overall user experience by supporting advanced communication formats beyond plain text. Voice calls, video calls, and multimedia sharing rely heavily on streaming technologies to transmit data seamlessly. For instance, during a video call within a chat app, continuous streaming of video and audio packets ensures the conversation is uninterrupted and synchronized.

Furthermore, streaming supports features like live translation, transcription, and reaction updates in real-time. The continuous flow of data allows these features to update dynamically without requiring users to refresh or reload content. This live interactivity enriches the chat experience, making it more engaging and intuitive.

### 3. Efficient Use of Network and Device Resources

Another significant advantage of streaming is its efficient use of network bandwidth and device resources. Instead of waiting for a complete message or media file to be generated before transmission, streaming sends data in smaller packets incrementally. This tokenized flow reduces peak bandwidth usage and spreads network load evenly over time, preventing bottlenecks.

Devices can start displaying or playing content as it arrives, which reduces memory and processing demands. For example, instead of downloading an entire voice note before playback, streaming allows users to listen while the remaining data continues to load. This reduces latency, avoids buffer underruns, and optimizes system performance — critical factors for mobile devices that have limited resources.

### 4. Scalability and Flexibility in Feature Implementation

Streaming also facilitates scalability and flexibility, allowing developers to implement a variety of interactive features and adapt to diverse communication scenarios. For example, chatbots and AI assistants integrated into chat apps can leverage streaming APIs to process user input continuously and output responses token-by-token. This interaction style is more user-friendly than waiting for the entire response to be generated, especially when responses are lengthy.

Similarly, streaming supports multi-modal input and output such as combining text, emojis, audio, and video, thereby enabling richer communication channels. It helps developers handle large volumes of concurrent connections by distributing data across streams rather than bulk transmissions, improving the app’s performance during peak usage periods.

### 5. Enabling Accessibility and Inclusivity

Streaming functionality in chat apps can also enhance accessibility for users with disabilities. Features such as real-time closed captioning and voice dictation depend on streaming audio data to generate live transcriptions or voice commands. This immediate feedback supports users who are hard of hearing or visually impaired by providing them with continuous updates and interaction without delay.

Additionally, streaming-based translation services can break language barriers in global chat applications by providing instant translated text or speech. This inclusivity expands the reach and usability of chat apps for diverse audiences.

---

### Conclusion

Streaming is a fundamental technological enabler in modern chat applications, delivering real-time interactivity, enhancing user experience, optimizing resource use, and supporting advanced features. Its ability to transmit data continuously and incrementally reduces latency, allows richer media integration, and scales efficiently across millions of users. By incorporating streaming, chat apps can provide seamless, engaging, and accessible communication experiences critical to personal and professional interactions worldwide.

## Streaming with the Responses API

Now we ask the same style of question, but receive output incrementally as events arrive.

In [5]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)

response_stream = client.responses.create(
    model=GENERATION_MODEL,
    instructions='You are a concise teaching assistant.',
    input='Explain in 3 paragraphse or more why streaming is useful in chat apps.',
    stream=True,
)

streamed_text = ''
handle = display(Markdown('_Streaming response will appear here..._'), display_id=True)

for event in response_stream:
    if event.type == 'response.output_text.delta':
        streamed_text += event.delta
        handle.update(Markdown(streamed_text))

Streaming is useful in chat apps because it enables real-time communication by sending data continuously as it becomes available. Instead of waiting for a complete message or a batch of messages to be prepared and transmitted, streaming allows each piece of data—such as text, images, or voice snippets—to flow instantly between users. This reduces latency, making conversations feel immediate and natural, which is crucial for maintaining engagement and responsiveness in chat applications.

Another key advantage of streaming in chat apps is efficient resource usage. Since data is transmitted incrementally, the app can start processing and displaying content without needing to load the entire message. This leads to faster updates and smoother user experience, especially in scenarios where messages are long or include media attachments. Streaming also minimizes buffering and helps maintain stable connections, which is particularly beneficial on networks with variable speeds or limited bandwidth.

Moreover, streaming supports advanced features like typing indicators, read receipts, and live media sharing by continuously exchanging small bits of information. This constant flow of data helps provide contextual cues, enhancing the overall communication experience by making interactions more interactive and dynamic. Consequently, streaming is a foundational technology that underpins the seamless, live nature of modern chat applications.

## Streaming with Ollama

Ollama can expose an OpenAI-compatible Responses API. That means the same event pattern can work against a local model server, as long as you already have a compatible model running locally.

Requires Ollama 0.13.3 or later for its stateless Responses API. Pull `llama3.2:latest` first; set `OLLAMA_CHAT_MODEL` to use another installed model.


In [6]:
# Optional: Ollama streaming with the OpenAI SDK and the Responses API
# This requires Ollama running locally and a model already pulled,
# for example: `ollama pull llama3.2:latest`.

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')
OLLAMA_CHAT_MODEL = os.getenv('OLLAMA_CHAT_MODEL', 'llama3.2:latest')

ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
ollama_stream = ollama_client.responses.create(
    model=OLLAMA_CHAT_MODEL,
    input='Explain in two short sentences why streaming is useful in chat apps.',
    stream=True,
)

ollama_text = ''
handle = display(Markdown('_Ollama streaming response will appear here..._'), display_id=True)

for event in ollama_stream:
    if event.type == 'response.output_text.delta':
        ollama_text += event.delta
        handle.update(Markdown(ollama_text))


Streaming allows users to share live video or audio content in chat apps, enabling real-time communication and social interaction. This feature is useful for applications such as live streaming of events, video conferencing, and gaming, where synchronous interaction is crucial.

## Streaming with Chat Completions

Compare OpenAI Responses events with Chat Completions deltas. This replaces the retired GitHub Models example. Empty housekeeping chunks may have no choices.


In [8]:
chat_client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)
chat_stream = chat_client.chat.completions.create(
    model=GENERATION_MODEL,
    messages=[{"role": "user", "content": "Explain in two short sentences why streaming is useful in chat apps."}],
    stream=True,
)
chat_text = ""
handle = display(Markdown("_Streaming..._"), display_id=True)
for chunk in chat_stream:
    if chunk.choices and chunk.choices[0].delta.content:
        chat_text += chunk.choices[0].delta.content
        handle.update(Markdown(chat_text))


Streaming in chat apps allows real-time delivery of messages, enabling instant communication between users. It also reduces latency and improves user experience by continuously updating the chat interface without needing page reloads.

Streaming is the common idea; the event format depends on the API. Responses emits typed events, while Chat Completions emits choices and content deltas.


## Turn the Event Loop into a Generator

UI frameworks often want a generator or iterable. That makes streaming a natural fit.

In [6]:
def stream_response(prompt: str, model: str = GENERATION_MODEL):
    client = OpenAI(api_key=OPENAI_API_KEY, organization=OPENAI_ORG_ID, project=OPENAI_PROJECT_ID)
    stream = client.responses.create(
        model=model,
        instructions='You are a concise teaching assistant.',
        input=prompt,
        stream=True,
    )

    for event in stream:
        if event.type == 'response.output_text.delta':
            yield event.delta

In [7]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

collected = []
handle = display(Markdown('_Streaming response will appear here..._'), display_id=True)

for piece in stream_response('Give one short paragraph on why streamed output feels faster to users.'):
    collected.append(piece)
    handle.update(Markdown(''.join(collected)))

full_text = ''.join(collected)

display(Markdown('### Final combined text'))
display(Markdown(full_text))

Streamed output feels faster to users because it provides immediate, incremental feedback rather than making them wait for the entire response. This continuous flow of information creates the impression of quick progress and keeps users engaged, reducing perceived latency and enhancing the overall user experience.

### Final combined text

Streamed output feels faster to users because it provides immediate, incremental feedback rather than making them wait for the entire response. This continuous flow of information creates the impression of quick progress and keeps users engaged, reducing perceived latency and enhancing the overall user experience.

## Why This Matters for the Rest of Session 8

- The Gradio demo can consume a generator like `stream_response(...)`.
- The Streamlit app can pass the same generator to `st.write_stream(...)`.
- Streaming is not a separate concept from app building; it is one of the main reasons the app feels interactive.